In [ ]:
import numpy as np
d = np.load("ml_ready/per_subject/P02_C_Place.npz", allow_pickle=True)
print(d.files)
X = d["X"]; print("X", X.shape, X.dtype, X.min(), X.max(), np.isnan(X).any())
print("y", d["y"].shape, np.unique(d["y"], return_counts=True))
print("subject", np.unique(d["subject"], return_counts=True))
print("sfreq", d["sfreq"], "ch", len(d["ch_names"]), "times", d["times"].shape)

## D2 probe — Place 자질이 EEG에 표상되나
DDA 계수 feature → SVM, permutation test로 우연 초과 판정. 불균형(69/25)이라 accuracy 대신 **balanced accuracy**.

In [ ]:
import numpy as np

d = np.load("ml_ready/per_subject/P02_C_Place.npz", allow_pickle=True)
print(d)
X = d["X"].astype(np.float64)
y = (d["y"] == "bilabial").astype(np.int64)
print("X", X.shape, "pos(bilabial)", int(y.sum()), "neg(alveolar)", int((1 - y).sum()))
print("majority baseline acc =", round(max(y.mean(), 1 - y.mean()), 3), "| chance bal-acc = 0.5")

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, permutation_test_score
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

d = np.load("ml_ready/per_subject/P02_C_Place.npz", allow_pickle=True)
X = d["X"].astype(np.float64)
y = (d["y"] == "bilabial").astype(np.int64)
print("X", X.shape, "pos(bilabial)", int(y.sum()), "neg(alveolar)", int((1 - y).sum()))
print("majority baseline acc =", round(max(y.mean(), 1 - y.mean()), 3), "| chance bal-acc = 0.5")

In [ ]:
def dda_coeffs(x: np.ndarray, tau1: int, tau2: int) -> np.ndarray:
    n, c, t = x.shape
    m = t - tau2 - 1
    out = np.empty((n, c, 3), dtype=np.float64)
    for i in range(n):
        for j in range(c):
            u = x[i, j]
            udot = u[tau2 + 1 : tau2 + 1 + m] - u[tau2 : tau2 + m]
            a = u[tau2 + 1 - tau1 : tau2 + 1 - tau1 + m]
            b = u[1 : 1 + m]
            reg = np.stack([a, b, a * a], axis=1)
            coef, *_ = np.linalg.lstsq(reg, udot, rcond=None)
            out[i, j] = coef
    return out.reshape(n, c * 3)

Xz = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-12)
F = dda_coeffs(Xz, tau1=7, tau2=10)
print("DDA feature", F.shape, "finite", np.isfinite(F).all())

In [ ]:
clf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

bal = cross_val_score(clf, F, y, cv=cv, scoring="balanced_accuracy")
auc = cross_val_score(clf, F, y, cv=cv, scoring="roc_auc")
print("balanced-acc  %.3f +- %.3f" % (bal.mean(), bal.std()))
print("roc-auc       %.3f +- %.3f" % (auc.mean(), auc.std()))

score, perm, pval = permutation_test_score(clf, F, y, cv=cv, scoring="balanced_accuracy", n_permutations=1000, random_state=0)
print("perm-test bal-acc %.3f  p=%.4f  (null %.3f +- %.3f)" % (score, pval, perm.mean(), perm.std()))
print("D2 verdict:", "ABOVE CHANCE (Place detectable)" if pval < 0.05 else "NOT above chance")

In [ ]:
for t1, t2 in [(3, 6), (5, 10), (7, 16), (10, 20), (12, 25)]:
    Ft = dda_coeffs(Xz, tau1=t1, tau2=t2)
    b = cross_val_score(clf, Ft, y, cv=cv, scoring="balanced_accuracy")
    print("tau (%2d,%2d)  bal-acc %.3f +- %.3f" % (t1, t2, b.mean(), b.std()))

## 진단 — "2명이라 안되나 vs feature/방법이 문제인가"
1) 자질축 전환(Place/Manner/Voicing)  2) feature 강화(잔차+다중window)  3) band-power baseline  4) learning curve  5) P01+P02 통합

In [ ]:
import pandas as pd

def load_subject(pid: str) -> tuple[np.ndarray, pd.DataFrame]:
    dd = np.load(f"ml_ready/per_subject/{pid}_C_Place.npz", allow_pickle=True)
    meta = pd.read_csv(f"ml_ready/per_subject/{pid}_C_Place.metadata.csv")
    return dd["X"].astype(np.float64), meta

X1, m1 = load_subject("P01")
X2, m2 = load_subject("P02")
Xall = np.concatenate([X1, X2], axis=0)
mall = pd.concat([m1, m2], ignore_index=True)
print("P01", X1.shape, "P02", X2.shape, "all", Xall.shape)
for col in ["Place", "Manner", "Voicing"]:
    print(col, dict(mall[col].value_counts()))

In [ ]:
def zscore(x: np.ndarray) -> np.ndarray:
    return (x - x.mean(axis=2, keepdims=True)) / (x.std(axis=2, keepdims=True) + 1e-12)

def dda_windowed(x: np.ndarray, tau1: int, tau2: int, win: int, hop: int) -> np.ndarray:
    n, c, t = x.shape
    starts = list(range(0, t - win + 1, hop))
    feats = []
    for s in starts:
        seg = x[:, :, s : s + win]
        m = win - tau2 - 1
        block = np.empty((n, c, 4), dtype=np.float64)
        for i in range(n):
            for j in range(c):
                u = seg[i, j]
                udot = u[tau2 + 1 : tau2 + 1 + m] - u[tau2 : tau2 + m]
                a = u[tau2 + 1 - tau1 : tau2 + 1 - tau1 + m]
                b = u[1 : 1 + m]
                reg = np.stack([a, b, a * a], axis=1)
                coef, res, *_ = np.linalg.lstsq(reg, udot, rcond=None)
                resid = float(np.mean((reg @ coef - udot) ** 2))
                block[i, j] = [coef[0], coef[1], coef[2], resid]
        feats.append(block)
    F = np.stack(feats, axis=1)
    mean = F.mean(axis=1)
    std = F.std(axis=1)
    return np.concatenate([mean, std], axis=2).reshape(n, -1)

Xz_all = zscore(Xall)
Fw = dda_windowed(Xz_all, tau1=7, tau2=10, win=128, hop=32)
print("windowed DDA feature", Fw.shape, "finite", np.isfinite(Fw).all())

In [ ]:
def band_power(x: np.ndarray, sfreq: float, bands: list[tuple[float, float]]) -> np.ndarray:
    from scipy.signal import welch
    n, c, t = x.shape
    f, P = welch(x, fs=sfreq, nperseg=min(128, t), axis=2)
    out = np.empty((n, c, len(bands)), dtype=np.float64)
    for k, (lo, hi) in enumerate(bands):
        sel = (f >= lo) & (f < hi)
        out[:, :, k] = np.log(P[:, :, sel].mean(axis=2) + 1e-20)
    return out.reshape(n, -1)

bands = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 60)]
Fbp = band_power(Xz_all, 256.0, bands)
print("band-power feature", Fbp.shape)

In [ ]:
clf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def probe(F: np.ndarray, y: np.ndarray, tag: str, nperm: int = 500) -> None:
    bal = cross_val_score(clf, F, y, cv=cv, scoring="balanced_accuracy")
    auc = cross_val_score(clf, F, y, cv=cv, scoring="roc_auc")
    s, _, p = permutation_test_score(clf, F, y, cv=cv, scoring="balanced_accuracy", n_permutations=nperm, random_state=0)
    flag = "**ABOVE**" if p < 0.05 else "chance"
    print("%-26s bal %.3f  auc %.3f  p=%.4f  %s" % (tag, bal.mean(), auc.mean(), p, flag))

for axis in ["Place", "Manner", "Voicing"]:
    vals = mall[axis].to_numpy()
    classes = np.unique(vals)
    if len(classes) != 2:
        print(axis, "skip (nclass=%d)" % len(classes)); continue
    yy = (vals == classes[1]).astype(np.int64)
    probe(Fw, yy, f"[DDA-win] {axis} ({classes[0]}/{classes[1]})")
    probe(Fbp, yy, f"[bandpow] {axis} ({classes[0]}/{classes[1]})")

In [ ]:
from sklearn.model_selection import learning_curve

yplace = (mall["Place"].to_numpy() == "bilabial").astype(np.int64)
sizes, tr, te = learning_curve(clf, Fw, yplace, cv=cv, scoring="balanced_accuracy", train_sizes=np.linspace(0.3, 1.0, 6), random_state=0)
print("train_size   test-bal-acc")
for s, row in zip(sizes, te):
    print("  %4d        %.3f +- %.3f" % (s, row.mean(), row.std()))
print("\n우상향이면 N늘리면 개선(데이터 부족), 평평하면 N문제 아님(feature/신호 문제)")

## 진단 시각화 — place 신호가 원신호/계수/공간에 있나
ERP · 채널별 t · DDA 계수분포 · PCA. 네 패널이 모두 겹치면 신호 부재, 단 채널평균이 공간패턴을 지우므로 CSP/SPD는 별도.

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA

place = mall["Place"].to_numpy()
alv = Xall[place == "alveolar"]
bil = Xall[place == "bilabial"]
tvec = d["times"]

fig, ax = plt.subplots(2, 2, figsize=(14, 10))

ea, eb = alv.mean((0, 1)), bil.mean((0, 1))
sa = alv.mean(1).std(0) / np.sqrt(len(alv))
sb = bil.mean(1).std(0) / np.sqrt(len(bil))
ax[0, 0].plot(tvec, ea, label=f"alveolar (n={len(alv)})", color="C0")
ax[0, 0].fill_between(tvec, ea - sa, ea + sa, alpha=0.2, color="C0")
ax[0, 0].plot(tvec, eb, label=f"bilabial (n={len(bil)})", color="C1")
ax[0, 0].fill_between(tvec, eb - sb, eb + sb, alpha=0.2, color="C1")
ax[0, 0].axhline(0, color="k", lw=0.5); ax[0, 0].set_title("Channel-mean ERP by Place")
ax[0, 0].set_xlabel("time (s)"); ax[0, 0].legend()

tv = np.array([stats.ttest_ind(alv[:, c].mean(1), bil[:, c].mean(1))[0] for c in range(Xall.shape[1])])
ax[0, 1].bar(range(len(tv)), tv, color=["C3" if abs(t) > 2 else "C7" for t in tv])
ax[0, 1].axhline(2, color="r", ls="--", lw=0.8); ax[0, 1].axhline(-2, color="r", ls="--", lw=0.8)
ax[0, 1].set_title(f"Per-channel t (max|t|={np.abs(tv).max():.2f}, sig={int((np.abs(tv)>2).sum())})")
ax[0, 1].set_xlabel("channel"); ax[0, 1].set_ylabel("t")

C = dda_coeffs(zscore(Xall), tau1=7, tau2=10).reshape(len(Xall), -1, 3).mean(1)
ya = place == "alveolar"
for k, nm in enumerate(["a1", "a2", "a3"]):
    ax[1, 0].hist(C[ya, k], bins=25, alpha=0.5, density=True, label=f"alv {nm}")
    ax[1, 0].hist(C[~ya, k], bins=25, alpha=0.5, density=True, label=f"bil {nm}")
ax[1, 0].set_title("DDA coeff dist by Place"); ax[1, 0].legend(fontsize=7)

pc = PCA(2).fit_transform(StandardScaler().fit_transform(C))
ax[1, 1].scatter(pc[ya, 0], pc[ya, 1], alpha=0.6, s=20, label="alveolar")
ax[1, 1].scatter(pc[~ya, 0], pc[~ya, 1], alpha=0.6, s=20, label="bilabial")
ax[1, 1].set_title("PCA of DDA coeffs"); ax[1, 1].legend()

plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt

In [17]:
import math

# ERP visualization for a .npz file
path = "PROJECT_ROOT/npz/sub-P01_ses-01.npz"
d = np.load(path, allow_pickle=True)
print("keys:", d.files)

X = d["X"].astype(np.float64)  # expect shape (n_trials, n_channels, n_times)
n_trials, n_ch, n_t = X.shape
times = d.get("times", None)
if times is None:
    sfreq = float(d.get("sfreq", 256.0))
    times = np.arange(n_t) / sfreq

ch_names = list(d.get("channel_names", [f"ch{i}" for i in range(n_ch)]))

mean_erp = X.mean(axis=0)           # (n_ch, n_t)
sem_erp = X.std(axis=0, ddof=0) / np.sqrt(n_trials)

ncols = 8
nrows = math.ceil(n_ch / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 1.8), sharex=True)
axes = np.array(axes).reshape(-1)

for i in range(n_ch):
    ax = axes[i]
    ax.plot(times, mean_erp[i], color="C0")
    ax.fill_between(times, mean_erp[i] - sem_erp[i], mean_erp[i] + sem_erp[i], color="C0", alpha=0.3)
    ax.axvline(0, color="k", lw=0.6, ls="--")
    ax.set_title(ch_names[i], fontsize=8)
    ax.tick_params(labelsize=7)
for ax in axes[n_ch:]:
    ax.axis("off")

fig.suptitle(f"ERP — {path.split('/')[-1]}  (n_trials={n_trials})", fontsize=12)
plt.xlabel("time (s)")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

keys: ['X', 'phoneme', 'place', 'category', 'manner', 'voicing', 'tms_target', 'tms_intensity', 'task', 'times', 'sfreq', 'ch_names']


In [ ]:
X.shape

In [3]:
import numpy as np
import pandas as pd

In [5]:
path_ori = "PROJECT_ROOT/npz/sub-P01_ses-01.npz"
dd = np.load(path_ori, allow_pickle=True)

In [6]:


pd = pd.DataFrame(dd)
pd

,0
0,X
1,phoneme
2,place
3,category
4,manner
5,voicing
6,tms_target
7,tms_intensity
8,task
9,times


In [11]:
path = "./eeg_data_processing_ses02_npz copy/per_subject_condition/s03_10_remove_bad_components_2_c.npz"
d = np.load(path, allow_pickle=True)

In [13]:
import pandas as pd
import numpy as np

pd = pd.DataFrame(d)
pd

,0
0,X
1,y_condition
2,condition_per_trial
3,subject_per_trial
4,epoch_in_file
5,event_code
6,event_label
7,event_id_json
8,channel_names
9,channel_types


In [14]:
pd.shape

(19, 1)

In [18]:
import warnings
warnings.filterwarnings("ignore")
import os
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.signal import find_peaks
import mne
mne.set_log_level("ERROR")

DEFAULT = "npz/sub-P01_ses-01.npz"


def load_epochs(path: str) -> tuple[np.ndarray, np.ndarray, dict[str, np.ndarray], list[str], float]:
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(float)
    t = d["times"]
    meta = {k: np.array([str(v) for v in d[k]]) for k in ["place", "tms_target", "phoneme"] if k in d}
    ch = [str(c) for c in d["channel_names"]]
    sf = float(d["sfreq"])
    base = t < 0
    Xb = X - X[:, :, base].mean(2, keepdims=True)
    return Xb, t, meta, ch, sf


def make_info(ch: list[str], sf: float) -> mne.Info:
    info = mne.create_info(ch, sf, "eeg")
    info.set_montage("standard_1020", match_case=False, on_missing="ignore")
    return info


def gfp(erp: np.ndarray) -> np.ndarray:
    return erp.std(axis=0)


def peak_latencies(t: np.ndarray, g: np.ndarray, n: int = 3) -> list[float]:
    w = (t >= 0.03) & (t <= 0.5)
    idx = np.where(w)[0]
    pk, _ = find_peaks(g[idx], distance=15)
    if len(pk) == 0:
        return [0.1, 0.2, 0.3]
    order = pk[np.argsort(g[idx][pk])[::-1][:n]]
    return sorted(float(t[idx][p]) for p in order)


def plot_erp(path: str, out: str, stops_only: bool = False) -> None:
    Xb, t, meta, ch, sf = load_epochs(path)
    info = make_info(ch, sf)
    if stops_only and "phoneme" in meta:
        keep = np.isin(meta["phoneme"], ["b", "p", "d", "t"])
        Xb = Xb[keep]
        meta = {k: v[keep] for k, v in meta.items()}
    ga = Xb.mean(0)
    g = gfp(ga)
    cz = ch.index("Cz") if "Cz" in ch else int(np.argmax(g.max(0)))
    mid = [c for c in ["Fz", "Cz", "Pz", "Oz"] if c in ch]
    lat = peak_latencies(t, g, 3)
    tm = t * 1000

    fig = plt.figure(figsize=(16, 11), facecolor="white")
    gs = GridSpec(3, 3, figure=fig, hspace=0.38, wspace=0.28)
    fig.suptitle("ERP overview  |  %s  |  %d trials, %d ch, %g Hz" % (
        os.path.basename(path), Xb.shape[0], Xb.shape[1], sf), fontsize=14, fontweight="bold")

    ax = fig.add_subplot(gs[0, :2])
    ax.plot(tm, ga.T, color="0.7", lw=0.4, alpha=0.6)
    ax.plot(tm, g, color="crimson", lw=2.2, label="GFP")
    for l in lat:
        ax.axvline(l * 1000, color="navy", ls="--", lw=1, alpha=0.6)
    ax.axvline(0, color="k", lw=1)
    ax.set_title("Butterfly (all channels) + GFP", fontsize=11)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Amplitude (a.u.)")
    ax.legend(loc="upper right", fontsize=9)

    axes_topo = [fig.add_axes([0.685 + i * 0.105, 0.70, 0.10, 0.16]) for i in range(len(lat))]
    for axi, l in zip(axes_topo, lat):
        ti = int(np.argmin(np.abs(t - l)))
        mne.viz.plot_topomap(ga[:, ti], info, axes=axi, show=False, cmap="RdBu_r", contours=4)
        axi.set_title("%d ms" % int(l * 1000), fontsize=9)

    ax = fig.add_subplot(gs[1, 0])
    for c in mid:
        ax.plot(tm, ga[ch.index(c)], lw=1.6, label=c)
    ax.axvline(0, color="k", lw=0.8); ax.axhline(0, color="0.6", lw=0.6)
    ax.set_title("Midline ERPs", fontsize=11); ax.set_xlabel("Time (ms)"); ax.set_ylabel("a.u.")
    ax.legend(fontsize=9)

    ax = fig.add_subplot(gs[1, 1])
    span = (t >= 0.08) & (t <= 0.2)
    order = np.argsort(np.abs(Xb[:, cz, span]).mean(1))
    vlim = np.percentile(np.abs(Xb[:, cz]), 98)
    im = ax.imshow(Xb[order, cz, :], aspect="auto", origin="lower", cmap="RdBu_r",
                   extent=[tm[0], tm[-1], 0, Xb.shape[0]], vmin=-vlim, vmax=vlim)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title("ERP image @ %s (trials x time)" % ch[cz], fontsize=11)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Trial (sorted)")
    fig.colorbar(im, ax=ax, fraction=0.046)

    ax = fig.add_subplot(gs[1, 2])
    for cond, col in [("alveolar", "C0"), ("bilabial", "C3")]:
        m = meta.get("place", np.array([])) == cond
        if m.sum() > 0:
            e = Xb[m, cz].mean(0); s = Xb[m, cz].std(0) / np.sqrt(m.sum())
            ax.plot(tm, e, color=col, lw=1.8, label="%s (n=%d)" % (cond, m.sum()))
            ax.fill_between(tm, e - s, e + s, color=col, alpha=0.2)
    ax.axvline(0, color="k", lw=0.8); ax.axhline(0, color="0.6", lw=0.6)
    ax.set_title("Place contrast @ %s (mean +- SEM)" % ch[cz], fontsize=11)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("a.u."); ax.legend(fontsize=9)

    ax = fig.add_subplot(gs[2, 0])
    groups = {"control": np.isin(meta.get("tms_target", np.array([])), ["control_lip", "control_tongue"]),
              "lip": meta.get("tms_target", np.array([])) == "lip",
              "tongue": meta.get("tms_target", np.array([])) == "tongue"}
    for (name, m), col in zip(groups.items(), ["k", "C1", "C2"]):
        if m.sum() > 0:
            ax.plot(tm, gfp(Xb[m].mean(0)), color=col, lw=1.8, label="%s (n=%d)" % (name, m.sum()))
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title("GFP by TMS condition", fontsize=11)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("GFP"); ax.legend(fontsize=9)

    ax = fig.add_subplot(gs[2, 1])
    vlim = np.percentile(np.abs(ga), 98)
    im = ax.imshow(ga, aspect="auto", origin="lower", cmap="RdBu_r",
                   extent=[tm[0], tm[-1], 0, len(ch)], vmin=-vlim, vmax=vlim)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title("Grand-average ERP (channels x time)", fontsize=11)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Channel")
    fig.colorbar(im, ax=ax, fraction=0.046)

    ax = fig.add_subplot(gs[2, 2])
    a = meta.get("place", np.array([])) == "alveolar"
    b = meta.get("place", np.array([])) == "bilabial"
    if a.sum() > 0 and b.sum() > 0:
        diff = Xb[a, cz].mean(0) - Xb[b, cz].mean(0)
        ax.plot(tm, diff, color="purple", lw=1.8)
        ax.fill_between(tm, 0, diff, color="purple", alpha=0.2)
    ax.axvline(0, color="k", lw=0.8); ax.axhline(0, color="0.6", lw=0.6)
    ax.set_title("Place difference wave (alv - bil) @ %s" % ch[cz], fontsize=11)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("a.u.")

    fig.savefig(out, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("saved:", out)
    print("GFP peaks (ms):", [int(l * 1000) for l in lat])


plot_erp("./eeg_data_processing_ses02_npz copy/per_subject_condition/s03_10_remove_bad_components_2_c.npz", "figs/erp_P02.png", stops_only=False)
plt.show()

saved: figs/erp_P02.png
GFP peaks (ms): [100, 200, 300]
